<a href="https://colab.research.google.com/github/Nazmaa-17/MLProject_EmotionDetection/blob/main/EmotionDetectionInTweets.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install -q datasets
import tensorflow as tf
from datasets import load_dataset
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Input, Embedding, LSTM, Dense, Dropout , Bidirectional
from tensorflow.keras.utils import to_categorical
import numpy as np

In [ ]:
dataset = load_dataset("dair-ai/emotion")

In [ ]:
vocab_size = 10000
max_length = 100
num_classes = 6

In [ ]:
train_texts = dataset['train']['text']
train_labels = dataset['train']['label']

val_texts = dataset['validation']['text']
val_labels = dataset['validation']['label']

test_texts = dataset['test']['text']
test_labels = dataset['test']['label']

In [ ]:
tokenizer = Tokenizer(num_words=vocab_size , oov_token="<UNK>")
tokenizer.fit_on_texts(train_texts)

In [ ]:
X_train = pad_sequences(tokenizer.texts_to_sequences(train_texts), maxlen=max_length)
X_val   = pad_sequences(tokenizer.texts_to_sequences(val_texts),   maxlen=max_length)
X_test  = pad_sequences(tokenizer.texts_to_sequences(test_texts),  maxlen=max_length)

In [ ]:
y_train = to_categorical(train_labels, num_classes=num_classes)
y_val   = to_categorical(val_labels,   num_classes=num_classes)
y_test  = to_categorical(test_labels,  num_classes=num_classes)

In [ ]:
emotion_map = {0: 'Sadness', 1: 'Joy', 2: 'Love', 3: 'Anger', 4: 'Fear', 5: 'Surprise'}

In [ ]:
word_index = tokenizer.word_index
reverse_word_index = {value: key for key, value in word_index.items()}

In [ ]:
def decode_text(encoded_sequence):
    return " ".join([reverse_word_index.get(i, "?") for i in encoded_sequence if i != 0])

In [ ]:
model = Sequential([
    Input (shape=(max_length,)),
    Embedding(input_dim=vocab_size, output_dim=128),
    Dropout(0.5),
    Bidirectional(LSTM(64, return_sequences=False)),
    Dense(32, activation='relu'),
    Dense(num_classes, activation='softmax')
])

In [ ]:
model.compile(
    loss='categorical_crossentropy',
    optimizer='adam',
    metrics=['accuracy']
)
model.summary()

In [ ]:
history = model.fit(
    X_train, y_train,
    validation_data=(X_val, y_val),
    epochs=5,
    batch_size=64
)
loss , accuracy = model.evaluate(X_test , y_test)
print(f"Test Accuracy : {accuracy}")

In [ ]:
def predict_emotion(text):

    encoded_text = tokenizer.texts_to_sequences([text])
    padded_text = pad_sequences(encoded_text, maxlen=max_length)

    prediction = model.predict(padded_text)
    predicted_class = np.argmax(prediction, axis=1)[0]

    return emotion_map.get(predicted_class, 'Unknown')

In [ ]:
test_samples = [
    "I am feeling very happy today and everything is going well.",
    "This is making me quite upset and angry.",
    "I am so sad to hear this news, it truly breaks my heart.",
    "This is amazing!",
    "I am so frustrated right now.",
    "I'm a little bit worried about what will happen next.",
]

for text in test_samples:
    print(f"Text: '{text}' -> Predicted Emotion: {predict_emotion(text)}")

In [ ]:
model.save('emotion_classifier_model.keras')
print("Model saved successfully to 'emotion_classifier_model.keras'")

In [ ]:
loaded_model = tf.keras.models.load_model('emotion_classifier_model.keras')
print("Model loaded successfully!\n")

test_samples = [
    "I am feeling very happy today and everything is going well.",
    "This is making me quite upset and angry.",
    "I am so sad to hear this news, it truly breaks my heart.",
    "This is amazing!",
    "I am so frustrated right now.",
    "I'm a little bit worried about what will happen next.",
]

print("--- Testing Loaded Model Predictions ---\n")

for text in test_samples:

    encoded_text = tokenizer.texts_to_sequences([text])
    padded_text = pad_sequences(encoded_text, maxlen=max_length)

    prediction_loaded = loaded_model.predict(padded_text, verbose=0)
    predicted_class_loaded = np.argmax(prediction_loaded, axis=1)[0]
    predicted_emotion_loaded = emotion_map.get(predicted_class_loaded, 'Unknown')

    print(f"Text: '{text}' -> Predicted Emotion: {predicted_emotion_loaded}")